# Results Visualization: Comprehensive Summary and Paper-Ready Figures

This notebook produces publication-quality summary figures combining all experimental results:

- **Table 1**: Attack metrics summary (ASR-R, ASR-A, ASR-T, Benign Accuracy)
- **Table 2**: Defense metrics summary (TPR, FPR, Precision, F1)
- **Figure 1**: Unified attack vs. defense effectiveness radar chart
- **Figure 2**: End-to-end threat model pipeline diagram
- **Figure 3**: Composite score comparison (attacks + defenses)
- **Figure 4**: 3D scatter — ASR-R, ASR-A, Benign Accuracy per attack
- **Figure 5**: Defense overhead vs. detection effectiveness
- **Figure 6**: Normalized defense effectiveness heatmap across attack families

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from evaluation.benchmarking import BenchmarkRunner, AttackEvaluator, DefenseEvaluator
from evaluation.retrieval_sim import RetrievalSimulator
from attacks.implementations import AttackSuite
from defenses.implementations import create_defense
from data.synthetic_corpus import SyntheticCorpus

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

SEED = 42
CORPUS_SIZE = 200
TOP_K = 5
N_POISON_BASE = 5

ATTACK_TYPES = ['agent_poison', 'minja', 'injecmem']
DEFENSE_TYPES = ['watermark', 'validation', 'proactive', 'composite']

ATTACK_LABELS = {
    'agent_poison': 'AgentPoison\n(Chen et al., 2024)',
    'minja': 'MINJA\n(Dong et al., 2025)',
    'injecmem': 'InjecMEM\n(ICLR 2026)',
}
ATTACK_LABELS_SHORT = {
    'agent_poison': 'AgentPoison',
    'minja': 'MINJA',
    'injecmem': 'InjecMEM',
}
DEFENSE_LABELS = {
    'watermark': 'Watermark\n(Zhao et al.)',
    'validation': 'Content\nValidation',
    'proactive': 'Proactive\nDefense',
    'composite': 'Composite\n(Multi-layer)',
}
DEFENSE_LABELS_SHORT = {
    'watermark': 'Watermark',
    'validation': 'Validation',
    'proactive': 'Proactive',
    'composite': 'Composite',
}

ATTACK_COLORS = {
    'agent_poison': '#E74C3C',
    'minja': '#3498DB',
    'injecmem': '#27AE60',
}
DEFENSE_COLORS = {
    'watermark': '#8E44AD',
    'validation': '#E67E22',
    'proactive': '#16A085',
    'composite': '#C0392B',
}

print('visualization setup complete')

## 1. Run All Experiments and Collect Results

In [ ]:
# run attack evaluations
sim = RetrievalSimulator(
    corpus_size=CORPUS_SIZE,
    top_k=TOP_K,
    n_poison_per_attack=N_POISON_BASE,
    seed=SEED,
)

attack_results = {}
for at in ATTACK_TYPES:
    m = sim.evaluate_attack(at)
    attack_results[at] = m
    print(f'{at}: ASR-R={m.asr_r:.3f}  ASR-A={m.asr_a:.3f}  ASR-T={m.asr_t:.3f}  Benign={m.benign_accuracy:.3f}')

# run defense evaluations
corpus = SyntheticCorpus(seed=SEED)
benign_entries = corpus.generate_benign_entries(50)
clean_content = [e['content'] for e in benign_entries]

attack_suite = AttackSuite()
poisoned_content = []
for entry in benign_entries[:20]:
    results = attack_suite.execute_all(entry['content'])
    for at, result in results['attack_results'].items():
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            poisoned_content.append(pc)

defense_evaluator = DefenseEvaluator()
defense_results = {}
for dt in DEFENSE_TYPES:
    m = defense_evaluator.evaluate_defense(dt, attack_suite, clean_content, poisoned_content)
    defense_results[dt] = m
    print(f'{dt}: TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  F1={m.f1_score:.3f}')

print(f'\nclean samples: {len(clean_content)}, poisoned samples: {len(poisoned_content)}')

## 2. Table 1: Attack Metrics Summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

table_data = []
for at in ATTACK_TYPES:
    m = attack_results[at]
    label = ATTACK_LABELS[at].replace('\n', ' ')
    table_data.append([
        label,
        f'{m.asr_r:.3f}',
        f'{m.asr_a:.3f}',
        f'{m.asr_t:.3f}',
        f'{m.benign_accuracy:.3f}',
        f'{m.asr_t:.3f}',  # threat score = ASR-T
    ])

col_labels = ['Attack', 'ASR-R', 'ASR-A', 'ASR-T', 'Benign Acc.', 'Threat Score']
col_colors = ['#ECF0F1'] * len(col_labels)
row_colors = [[ATTACK_COLORS[at] + '33'] * len(col_labels) for at in ATTACK_TYPES]

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(11)

# style header
for j, label in enumerate(col_labels):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# style rows
for i, at in enumerate(ATTACK_TYPES):
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(row_colors[i][j])
    table[i+1, 0].set_text_props(fontweight='bold')

ax.set_title('Table 1: Attack Methodology Evaluation Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualization/fig_table1_attack_summary.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_table1_attack_summary.pdf', bbox_inches='tight')
plt.show()
print('Table 1 saved')

## 3. Table 2: Defense Metrics Summary

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.axis('off')

table_data = []
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    label = DEFENSE_LABELS[dt].replace('\n', ' ')
    youden = m.tpr - m.fpr
    table_data.append([
        label,
        f'{m.tpr:.3f}',
        f'{m.fpr:.3f}',
        f'{m.precision:.3f}',
        f'{m.f1_score:.3f}',
        f'{youden:.3f}',
        f'{m.true_positives}',
        f'{m.false_positives}',
    ])

col_labels = ['Defense', 'TPR', 'FPR', 'Precision', 'F1', 'Youden J', 'TP', 'FP']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(10)

for j in range(len(col_labels)):
    table[0, j].set_facecolor('#1A252F')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i, dt in enumerate(DEFENSE_TYPES):
    m = defense_results[dt]
    # color by F1: green=high, yellow=medium, red=low
    if m.f1_score >= 0.7:
        row_bg = '#D5F5E3'
    elif m.f1_score >= 0.4:
        row_bg = '#FEF9E7'
    else:
        row_bg = '#FDEDEC'
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(row_bg)
    table[i+1, 0].set_text_props(fontweight='bold')

ax.set_title('Table 2: Defense Mechanism Evaluation Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../visualization/fig_table2_defense_summary.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_table2_defense_summary.pdf', bbox_inches='tight')
plt.show()
print('Table 2 saved')

## 4. Figure 1: Radar Chart — Attack vs. Defense Effectiveness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(polar=True))
fig.suptitle('Figure 1: Radar Profile — Attack Effectiveness and Defense Coverage', fontsize=13, fontweight='bold')

# --- attack radar ---
attack_metric_labels = ['ASR-R', 'ASR-A', 'ASR-T', 'Benign\nAcc.', 'Stealthiness']
n_attack = len(attack_metric_labels)
angles_a = np.linspace(0, 2 * np.pi, n_attack, endpoint=False).tolist()
angles_a += angles_a[:1]

ax_a = axes[0]
ax_a.set_theta_offset(np.pi / 2)
ax_a.set_theta_direction(-1)
ax_a.set_thetagrids(np.degrees(angles_a[:-1]), attack_metric_labels)
ax_a.set_title('Attack Profiles', size=12, pad=15)

for at in ATTACK_TYPES:
    m = attack_results[at]
    # stealthiness = benign_accuracy (high = stealthy)
    values = [m.asr_r, m.asr_a, m.asr_t, m.benign_accuracy, m.benign_accuracy]
    values += values[:1]
    ax_a.plot(angles_a, values, 'o-', linewidth=2, color=ATTACK_COLORS[at],
              label=ATTACK_LABELS_SHORT[at])
    ax_a.fill(angles_a, values, alpha=0.1, color=ATTACK_COLORS[at])

ax_a.set_ylim(0, 1)
ax_a.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_a.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax_a.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=9)

# --- defense radar ---
defense_metric_labels = ['TPR', 'Precision', 'F1', '1-FPR\n(Specificity)', 'Youden J']
n_defense = len(defense_metric_labels)
angles_d = np.linspace(0, 2 * np.pi, n_defense, endpoint=False).tolist()
angles_d += angles_d[:1]

ax_d = axes[1]
ax_d.set_theta_offset(np.pi / 2)
ax_d.set_theta_direction(-1)
ax_d.set_thetagrids(np.degrees(angles_d[:-1]), defense_metric_labels)
ax_d.set_title('Defense Profiles', size=12, pad=15)

for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    youden = m.tpr - m.fpr
    values = [m.tpr, m.precision, m.f1_score, 1.0 - m.fpr, max(0.0, youden)]
    values += values[:1]
    ax_d.plot(angles_d, values, 'o-', linewidth=2, color=DEFENSE_COLORS[dt],
              label=DEFENSE_LABELS_SHORT[dt])
    ax_d.fill(angles_d, values, alpha=0.1, color=DEFENSE_COLORS[dt])

ax_d.set_ylim(0, 1)
ax_d.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_d.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax_d.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout()
plt.savefig('../visualization/fig01_radar_attack_defense.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig01_radar_attack_defense.pdf', bbox_inches='tight')
plt.show()
print('Figure 1 saved: fig01_radar_attack_defense')

## 5. Figure 2: Threat Model Pipeline Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#FAFAFA')
fig.patch.set_facecolor('#FAFAFA')

def draw_box(ax, x, y, w, h, label, sublabel=None, color='#3498DB', text_color='white', fontsize=10):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='#2C3E50',
                          linewidth=1.5, zorder=3, alpha=0.9)
    ax.add_patch(rect)
    if sublabel:
        ax.text(x + w/2, y + h*0.62, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)
        ax.text(x + w/2, y + h*0.3, sublabel, ha='center', va='center',
                fontsize=fontsize - 1.5, color=text_color, alpha=0.9, zorder=4)
    else:
        ax.text(x + w/2, y + h/2, label, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, color='#2C3E50', label=None):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0), zorder=2)
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2 + 0.15
        ax.text(mx, my, label, ha='center', va='bottom', fontsize=8.5,
                color=color, style='italic')

# adversary section
adv_bg = plt.Rectangle((0.3, 4.2), 5.2, 2.4, facecolor='#FDEDEC', edgecolor='#E74C3C',
                         linewidth=2, linestyle='--', zorder=1, alpha=0.5)
ax.add_patch(adv_bg)
ax.text(0.6, 6.3, 'Adversary', fontsize=9, color='#E74C3C', fontweight='bold')

draw_box(ax, 0.5, 4.4, 2.2, 1.8, 'Attack\nGeneration', 'AgentPoison / MINJA / InjecMEM',
         color='#E74C3C', fontsize=9)
draw_box(ax, 3.2, 4.4, 2.0, 1.8, 'Poison\nPassages', 'trigger-optimized content',
         color='#C0392B', fontsize=9)

draw_arrow(ax, 2.72, 5.3, 3.2, 5.3)

# memory system section
mem_bg = plt.Rectangle((5.8, 1.0), 4.4, 5.5, facecolor='#EBF5FB', edgecolor='#3498DB',
                         linewidth=2, linestyle='--', zorder=1, alpha=0.5)
ax.add_patch(mem_bg)
ax.text(6.0, 6.2, 'Agent Memory System', fontsize=9, color='#3498DB', fontweight='bold')

draw_box(ax, 6.0, 4.4, 3.8, 1.8, 'Vector Store', 'FAISS + sentence-transformers',
         color='#2980B9', fontsize=9)
draw_box(ax, 6.0, 1.2, 3.8, 1.8, 'Benign Corpus', f'N={CORPUS_SIZE} synthetic agent memory entries',
         color='#5DADE2', fontsize=9)

draw_arrow(ax, 7.9, 4.4, 7.9, 3.0, label='inject')

# injection arrow from adversary to memory
draw_arrow(ax, 5.2, 5.3, 6.0, 5.3, color='#E74C3C', label='poison injection')

# defense section
def_bg = plt.Rectangle((0.3, 0.3), 5.2, 3.5, facecolor='#E8F8F5', edgecolor='#27AE60',
                         linewidth=2, linestyle='--', zorder=1, alpha=0.5)
ax.add_patch(def_bg)
ax.text(0.6, 3.5, 'Defense Layer', fontsize=9, color='#27AE60', fontweight='bold')

draw_box(ax, 0.5, 0.5, 2.0, 1.5, 'Watermark\nDefense', 'z-score detection',
         color='#8E44AD', fontsize=8.5)
draw_box(ax, 3.0, 0.5, 2.2, 1.5, 'Content\nValidation', 'pattern filtering',
         color='#E67E22', fontsize=8.5)
draw_box(ax, 0.5, 2.2, 2.0, 1.2, 'Proactive\nDefense', 'pre-emptive scan',
         color='#16A085', fontsize=8.5)
draw_box(ax, 3.0, 2.2, 2.2, 1.2, 'Composite\nDefense', 'multi-layer fusion',
         color='#C0392B', fontsize=8.5)

# victim query section
draw_box(ax, 11.0, 4.4, 2.8, 1.8, 'Victim Agent', 'issues target queries',
         color='#F39C12', fontsize=9)

# evaluation
draw_box(ax, 11.0, 1.2, 2.8, 1.8, 'Evaluation', 'ASR-R / ASR-A / ASR-T\nTPR / FPR / F1',
         color='#1ABC9C', fontsize=8.5)

# arrows
draw_arrow(ax, 11.0, 5.3, 9.8, 5.3, color='#F39C12', label='query retrieval')
draw_arrow(ax, 9.8, 4.4, 9.8, 3.0, color='#2980B9', label='top-k results')
draw_arrow(ax, 9.8, 2.0, 11.0, 2.1, color='#1ABC9C', label='metrics')
draw_arrow(ax, 5.5, 1.5, 6.0, 2.5, color='#27AE60', label='detect')

ax.set_title('Figure 2: End-to-End Threat Model — Memory Poisoning Attack and Defense Pipeline',
             fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('../visualization/fig02_threat_model_pipeline.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig02_threat_model_pipeline.pdf', bbox_inches='tight')
plt.show()
print('Figure 2 saved: fig02_threat_model_pipeline')

## 6. Figure 3: Composite Score Comparison (Attack Threat + Defense Coverage)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 3: Composite Threat and Coverage Scores', fontsize=13, fontweight='bold')

# left: attack composite threat score (normalized weighted sum)
# threat = 0.4*ASR_R + 0.3*ASR_A + 0.3*ASR_T
ax_l = axes[0]
threat_scores = []
for at in ATTACK_TYPES:
    m = attack_results[at]
    threat = 0.4 * m.asr_r + 0.3 * m.asr_a + 0.3 * m.asr_t
    threat_scores.append(threat)

x = np.arange(len(ATTACK_TYPES))
bars = ax_l.bar(x, threat_scores, 0.5,
                color=[ATTACK_COLORS[at] for at in ATTACK_TYPES],
                edgecolor='black', linewidth=0.6, alpha=0.87)
for bar, val in zip(bars, threat_scores):
    ax_l.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
              f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax_l.set_xticks(x)
ax_l.set_xticklabels([ATTACK_LABELS[at] for at in ATTACK_TYPES], fontsize=9)
ax_l.set_ylabel('Composite Threat Score')
ax_l.set_title('Attack Threat Score\n(0.4·ASR-R + 0.3·ASR-A + 0.3·ASR-T)', fontsize=10)
ax_l.set_ylim(0, 1.0)
ax_l.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)

# right: defense composite coverage score
# coverage = 0.4*F1 + 0.3*TPR + 0.3*(1-FPR)
ax_r = axes[1]
coverage_scores = []
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    coverage = 0.4 * m.f1_score + 0.3 * m.tpr + 0.3 * (1.0 - m.fpr)
    coverage_scores.append(coverage)

x2 = np.arange(len(DEFENSE_TYPES))
bars2 = ax_r.bar(x2, coverage_scores, 0.5,
                 color=[DEFENSE_COLORS[dt] for dt in DEFENSE_TYPES],
                 edgecolor='black', linewidth=0.6, alpha=0.87)
for bar, val in zip(bars2, coverage_scores):
    ax_r.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
              f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax_r.set_xticks(x2)
ax_r.set_xticklabels([DEFENSE_LABELS[dt] for dt in DEFENSE_TYPES], fontsize=9)
ax_r.set_ylabel('Composite Coverage Score')
ax_r.set_title('Defense Coverage Score\n(0.4·F1 + 0.3·TPR + 0.3·Specificity)', fontsize=10)
ax_r.set_ylim(0, 1.0)
ax_r.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.savefig('../visualization/fig03_composite_scores.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig03_composite_scores.pdf', bbox_inches='tight')
plt.show()
print('Figure 3 saved: fig03_composite_scores')

## 7. Figure 4: 3D Scatter — ASR-R, ASR-A, Benign Accuracy

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

attack_markers_3d = {'agent_poison': 'o', 'minja': 's', 'injecmem': '^'}

for at in ATTACK_TYPES:
    m = attack_results[at]
    ax.scatter(
        m.asr_r, m.asr_a, m.benign_accuracy,
        c=ATTACK_COLORS[at],
        marker=attack_markers_3d[at],
        s=300,
        zorder=5,
        label=ATTACK_LABELS[at].replace('\n', ' '),
        edgecolors='black',
        linewidths=1,
        alpha=0.9,
    )
    ax.text(m.asr_r + 0.02, m.asr_a + 0.02, m.benign_accuracy + 0.01,
            ATTACK_LABELS_SHORT[at], fontsize=9, fontweight='bold')

# draw projections (dashed lines to axes planes)
for at in ATTACK_TYPES:
    m = attack_results[at]
    ax.plot([m.asr_r, m.asr_r], [m.asr_a, m.asr_a], [0, m.benign_accuracy],
            '--', color=ATTACK_COLORS[at], alpha=0.3, linewidth=0.8)
    ax.plot([m.asr_r, m.asr_r], [0, m.asr_a], [m.benign_accuracy, m.benign_accuracy],
            '--', color=ATTACK_COLORS[at], alpha=0.3, linewidth=0.8)
    ax.plot([0, m.asr_r], [m.asr_a, m.asr_a], [m.benign_accuracy, m.benign_accuracy],
            '--', color=ATTACK_COLORS[at], alpha=0.3, linewidth=0.8)

ax.set_xlabel('ASR-R (Retrieval Rate)', fontsize=10, labelpad=10)
ax.set_ylabel('ASR-A (Action Rate | Retrieval)', fontsize=10, labelpad=10)
ax.set_zlabel('Benign Accuracy (Stealthiness)', fontsize=10, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.set_title('Figure 4: 3D Attack Profile — Retrieval, Action, and Stealthiness Dimensions',
             fontsize=11, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=9, bbox_to_anchor=(0.0, 1.05))
ax.view_init(elev=20, azim=225)

plt.tight_layout()
plt.savefig('../visualization/fig04_3d_attack_profile.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig04_3d_attack_profile.pdf', bbox_inches='tight')
plt.show()
print('Figure 4 saved: fig04_3d_attack_profile')

## 8. Figure 5: Defense Utility — TPR vs. (1 - FPR) Pareto Frontier

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

defense_markers = {'watermark': 'o', 'validation': 's', 'proactive': '^', 'composite': 'D'}

# theoretical optimal and random lines
ax.plot([0, 1], [1, 1], 'g--', alpha=0.3, linewidth=1, label='Ideal TPR=1')
ax.plot([1, 1], [0, 1], 'g--', alpha=0.3, linewidth=1)
ax.fill_between([0.7, 1.0], [0.7, 0.7], [1.0, 1.0], alpha=0.07, color='green',
                label='High-performance region')

# F1 iso-contours
specificity_vals = np.linspace(0.01, 1.0, 300)
for f1_target in [0.3, 0.5, 0.7, 0.9]:
    # F1 = 2*P*R / (P+R) — simplified here as contour in TPR vs specificity space
    # approximate: plot as horizontal dashed contour at TPR = f1_target
    ax.axhline(y=f1_target, color='lightgray', linestyle=':', alpha=0.6, linewidth=0.8)
    ax.text(0.02, f1_target + 0.01, f'TPR={f1_target:.1f}', fontsize=7.5, color='gray')

# plot defenses in TPR vs. Specificity (1 - FPR) space
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    specificity = 1.0 - m.fpr
    ax.scatter(
        specificity, m.tpr,
        c=DEFENSE_COLORS[dt],
        marker=defense_markers[dt],
        s=220,
        zorder=5,
        label=f'{DEFENSE_LABELS_SHORT[dt]} (F1={m.f1_score:.2f}, J={m.tpr - m.fpr:.2f})',
        edgecolors='black',
        linewidths=1.2,
    )
    # annotate
    ax.annotate(
        f'  {DEFENSE_LABELS_SHORT[dt]}',
        (specificity, m.tpr),
        fontsize=9, fontweight='bold', color=DEFENSE_COLORS[dt],
    )

ax.set_xlabel('Specificity (1 - FPR)', fontsize=12)
ax.set_ylabel('Sensitivity (TPR)', fontsize=12)
ax.set_title('Figure 5: Defense Pareto Frontier — Sensitivity vs. Specificity',
             fontsize=12, fontweight='bold')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.legend(loc='lower left', fontsize=9)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../visualization/fig05_defense_pareto_frontier.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig05_defense_pareto_frontier.pdf', bbox_inches='tight')
plt.show()
print('Figure 5 saved: fig05_defense_pareto_frontier')

## 9. Figure 6: Normalized Defense Effectiveness Heatmap Across Attack Families

In [ ]:
# compute per-attack defense effectiveness using attack-specific poisoned content
per_attack_defense_matrix = np.zeros((len(DEFENSE_TYPES), len(ATTACK_TYPES)))

for j, at in enumerate(ATTACK_TYPES):
    # generate poisoned content specific to this attack type
    atk = attack_suite.get_attack(at)
    attack_poison = []
    for entry in benign_entries[:15]:
        result = atk.execute(entry['content'])
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            attack_poison.append(pc)

    for i, dt in enumerate(DEFENSE_TYPES):
        m = defense_evaluator.evaluate_defense(
            dt, attack_suite, clean_content[:15], attack_poison
        )
        per_attack_defense_matrix[i, j] = m.f1_score

# normalize each column (attack family) to [0, 1] across defenses
normalized_matrix = np.zeros_like(per_attack_defense_matrix)
for j in range(len(ATTACK_TYPES)):
    col = per_attack_defense_matrix[:, j]
    col_min, col_max = col.min(), col.max()
    if col_max > col_min:
        normalized_matrix[:, j] = (col - col_min) / (col_max - col_min)
    else:
        normalized_matrix[:, j] = 0.5

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Figure 6: Defense F1 Effectiveness Across Attack Families',
             fontsize=13, fontweight='bold')

# raw F1 heatmap
ax1 = axes[0]
im1 = ax1.imshow(per_attack_defense_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax1.set_xticks(range(len(ATTACK_TYPES)))
ax1.set_xticklabels([ATTACK_LABELS_SHORT[at] for at in ATTACK_TYPES])
ax1.set_yticks(range(len(DEFENSE_TYPES)))
ax1.set_yticklabels([DEFENSE_LABELS_SHORT[dt] for dt in DEFENSE_TYPES])
ax1.set_title('Raw F1 Score', fontsize=11)
plt.colorbar(im1, ax=ax1, label='F1 Score')
for i in range(len(DEFENSE_TYPES)):
    for j in range(len(ATTACK_TYPES)):
        ax1.text(j, i, f'{per_attack_defense_matrix[i, j]:.2f}',
                 ha='center', va='center', fontsize=11, fontweight='bold',
                 color='white' if per_attack_defense_matrix[i, j] < 0.4 else 'black')

# normalized heatmap (relative effectiveness within each attack family)
ax2 = axes[1]
im2 = ax2.imshow(normalized_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(ATTACK_TYPES)))
ax2.set_xticklabels([ATTACK_LABELS_SHORT[at] for at in ATTACK_TYPES])
ax2.set_yticks(range(len(DEFENSE_TYPES)))
ax2.set_yticklabels([DEFENSE_LABELS_SHORT[dt] for dt in DEFENSE_TYPES])
ax2.set_title('Normalized (per Attack Family)', fontsize=11)
plt.colorbar(im2, ax=ax2, label='Relative Effectiveness')
for i in range(len(DEFENSE_TYPES)):
    for j in range(len(ATTACK_TYPES)):
        ax2.text(j, i, f'{normalized_matrix[i, j]:.2f}',
                 ha='center', va='center', fontsize=11, fontweight='bold',
                 color='white' if normalized_matrix[i, j] < 0.4 else 'black')

plt.tight_layout()
plt.savefig('../visualization/fig06_defense_effectiveness_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig06_defense_effectiveness_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Figure 6 saved: fig06_defense_effectiveness_heatmap')

## 10. Save Consolidated Results

In [ ]:
import random as _random

# simulate minja isr across base success probabilities (n_entries=100, n_turns=3)
base_probs = np.linspace(0.50, 1.00, 30)
n_entries = 100
n_turns = 3
shortening_rate = 0.10

def simulate_isr(base_prob, n_entries, n_turns, shortening_rate, seed=42):
    """simulate minja multi-turn isr for a given base probability."""
    rng = _random.Random(seed)
    successful = 0
    for _ in range(n_entries):
        injected = False
        for turn_idx in range(n_turns):
            turn_prob = base_prob * max(0.5, 1.0 - shortening_rate * turn_idx)
            if rng.random() < turn_prob:
                injected = True
                break
        if injected:
            successful += 1
    return successful / n_entries

isr_1turn = [simulate_isr(p, n_entries, 1, shortening_rate) for p in base_probs]
isr_2turn = [simulate_isr(p, n_entries, 2, shortening_rate) for p in base_probs]
isr_3turn = [simulate_isr(p, n_entries, 3, shortening_rate) for p in base_probs]
isr_5turn = [simulate_isr(p, n_entries, 5, shortening_rate) for p in base_probs]

# paper-reported point: base_prob ≈ 0.98, isr ≈ 0.982 with 3 turns
paper_prob, paper_isr = 0.98, 0.982

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Figure 10a: MINJA Multi-Turn Injection Success Rate (ISR) Simulation\n'
             '(Dong et al., NeurIPS 2025 — Progressive Shortening Protocol)', fontsize=12, fontweight='bold')

# left: isr vs base success probability for different turn counts
ax = axes[0]
ax.plot(base_probs, isr_1turn, color='#BDC3C7', linewidth=2, label='n_turns=1', linestyle='--')
ax.plot(base_probs, isr_2turn, color='#85C1E9', linewidth=2, label='n_turns=2')
ax.plot(base_probs, isr_3turn, color='#3498DB', linewidth=2.5, label='n_turns=3 (paper)')
ax.plot(base_probs, isr_5turn, color='#1A5276', linewidth=2, label='n_turns=5')
ax.scatter([paper_prob], [paper_isr], color='red', s=150, zorder=5,
           label=f'Paper (ISR={paper_isr:.3f})', marker='*')
ax.axhline(y=paper_isr, color='red', linestyle=':', alpha=0.5, linewidth=1)
ax.axvline(x=paper_prob, color='red', linestyle=':', alpha=0.5, linewidth=1)
ax.set_xlabel('Base Injection Success Probability (p₀)')
ax.set_ylabel('Injection Success Rate (ISR)')
ax.set_title('ISR vs. Base Success Probability\nfor Different Indication Turn Counts')
ax.legend(fontsize=9)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.25)

# right: isr vs shortening rate (at base_prob=0.98, n_turns=3)
shortening_rates = np.linspace(0.0, 0.5, 25)
isr_by_shortening = [simulate_isr(0.98, n_entries, 3, sr) for sr in shortening_rates]
ax = axes[1]
ax.plot(shortening_rates, isr_by_shortening, color='#3498DB', linewidth=2.5, marker='o',
        markersize=4, label='ISR (base_prob=0.98, n_turns=3)')
ax.axhline(y=paper_isr, color='red', linestyle=':', alpha=0.7, linewidth=1.5,
           label=f'Paper ISR = {paper_isr:.3f}')
ax.axvline(x=0.10, color='gray', linestyle='--', alpha=0.7, linewidth=1.5,
           label='Paper shortening rate = 0.10')
ax.fill_between(shortening_rates, isr_by_shortening, paper_isr,
                where=[v < paper_isr for v in isr_by_shortening],
                alpha=0.15, color='orange', label='ISR gap from paper')
ax.set_xlabel('Progressive Shortening Rate per Turn')
ax.set_ylabel('Injection Success Rate (ISR)')
ax.set_title('ISR vs. Shortening Rate\n(base_prob=0.98, n_turns=3, n_entries=100)')
ax.legend(fontsize=9)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('../visualization/fig10a_minja_isr_simulation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig10a_minja_isr_simulation.pdf', bbox_inches='tight')
plt.show()
print('Figure 10a saved: fig10a_minja_isr_simulation')

## 10. Phase 10 Upgrade: MINJA ISR Simulation and AgentPoison Trigger Optimization Comparison

Detailed visualization of the Phase 10 improvements:
- **MINJA ISR simulation**: multi-turn injection success rate across varying base probabilities and indication turns
- **AgentPoison ASR-R improvement**: trigger-optimized vs. query-echo baseline comparison

In [ ]:
from dataclasses import asdict

consolidated = {
    'experiment': 'consolidated_results_visualization',
    'corpus_size': CORPUS_SIZE,
    'top_k': TOP_K,
    'n_poison_base': N_POISON_BASE,
    'seed': SEED,
    'attack_metrics': {at: asdict(attack_results[at]) for at in ATTACK_TYPES},
    'defense_metrics': {dt: asdict(defense_results[dt]) for dt in DEFENSE_TYPES},
    'composite_threat_scores': {
        at: round(
            0.4 * attack_results[at].asr_r +
            0.3 * attack_results[at].asr_a +
            0.3 * attack_results[at].asr_t, 4
        )
        for at in ATTACK_TYPES
    },
    'composite_coverage_scores': {
        dt: round(
            0.4 * defense_results[dt].f1_score +
            0.3 * defense_results[dt].tpr +
            0.3 * (1.0 - defense_results[dt].fpr), 4
        )
        for dt in DEFENSE_TYPES
    },
    'per_attack_defense_f1_matrix': per_attack_defense_matrix.tolist(),
    'normalized_defense_matrix': normalized_matrix.tolist(),
    'attack_order': ATTACK_TYPES,
    'defense_order': DEFENSE_TYPES,
}

output_path = '../visualization/consolidated_results.json'
with open(output_path, 'w') as f:
    json.dump(consolidated, f, indent=2)

print(f'Consolidated results saved to {output_path}')
print('\n=== Attack Summary ===')
for at in ATTACK_TYPES:
    m = attack_results[at]
    ts = consolidated['composite_threat_scores'][at]
    print(f'{at:15s}: ASR-R={m.asr_r:.3f}  ASR-T={m.asr_t:.3f}  Benign={m.benign_accuracy:.3f}  Threat={ts:.3f}')

print('\n=== Defense Summary ===')
for dt in DEFENSE_TYPES:
    m = defense_results[dt]
    cs = consolidated['composite_coverage_scores'][dt]
    print(f'{dt:15s}: TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  F1={m.f1_score:.3f}  Coverage={cs:.3f}')